In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score
CKPT_DIR = "/content/drive/MyDrive/llm_results"

Mounted at /content/drive


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from google.colab import files
import os
import torch

!pip install -q transformers accelerate bitsandbytes

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


In [ ]:
N_PER_SOURCE, SEED = 600, 17
parts = []
for strat in ("random", "margin", "entropy"):
    a = pd.read_csv(f"{CKPT_DIR}/sentigold_llm_annotator_{strat}_annotations.csv")
    parts.append(a.sample(N_PER_SOURCE, random_state=SEED).assign(source=strat))
bench = pd.concat(parts, ignore_index=True).drop_duplicates(subset="sentence").reset_index(drop=True)
bench = bench.rename(columns={"true_label": "gold", "train_label": "zs_label", "annotator_conf": "zs_conf"})
bench = bench[["sentence", "gold", "zs_label", "zs_conf", "source"]]
bench.to_csv(f"{CKPT_DIR}/annotator_benchmark.csv", index=False)

names = ["neutral", "positive", "negative"]
print("benchmark rows:", len(bench), "| per source:", bench.source.value_counts().to_dict())
for src, g in [("ALL", bench)] + list(bench.groupby("source")):
    print(f"{src:8s} n={len(g):4d} acc={accuracy_score(g.gold, g.zs_label):.3f} "
          f"macroF1={f1_score(g.gold, g.zs_label, average='macro'):.3f} "
          f"LLM says neutral {np.mean(g.zs_label == 0):.0%} (gold {np.mean(g.gold == 0):.0%})")

benchmark rows: 1707 | per source: {'random': 600, 'margin': 563, 'entropy': 544}
ALL      n=1707 acc=0.617 macroF1=0.616 LLM says neutral 42% (gold 23%)
entropy  n= 544 acc=0.568 macroF1=0.560 LLM says neutral 40% (gold 18%)
margin   n= 563 acc=0.570 macroF1=0.569 LLM says neutral 46% (gold 23%)
random   n= 600 acc=0.705 macroF1=0.706 LLM says neutral 41% (gold 26%)


In [ ]:
# ===== DROP-IN REPLACEMENT for the BASE_URL / read_csv cell =====
# Produces train_raw, val_raw, test_raw with columns "Data" and "Label" (0=neutral, 1=positive, 2=negative),
# with the column names the rest of the notebook expects.
!pip install -q datasets
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 17
TOTAL = 15700          # subsample size (about 15.7k rows); set None to use everything

ds = load_dataset("SayedShaun/sentigold")
raw = ds["train"].to_pandas()
print("columns:", raw.columns.tolist())
print(raw["Polarity"].value_counts())

# 5 -> 3 classes, same ids as the notebook's LABEL_MAP (neutral=0, positive=1, negative=2)
COLLAPSE = {"SP": 1, "WP": 1, "NU": 0, "WN": 2, "SN": 2}
unknown = set(raw["Polarity"].unique()) - set(COLLAPSE)
assert not unknown, f"Unexpected Polarity values {unknown}: edit COLLAPSE using the counts printed above"

df = pd.DataFrame({"Data": raw["Text"], "Label": raw["Polarity"].map(COLLAPSE)})
if "Domain" in raw.columns:
    df["Domain"] = raw["Domain"]          # extra column, ignored by clean(); handy for later analysis

# pre-apply the notebook's own filters (non-null, 1 < length < 100, no duplicates) so clean() doesn't shrink it much
df = df.dropna(subset=["Data", "Label"])
df["Data"] = df["Data"].astype(str).str.strip()
df = df[(df["Data"].str.len() > 1) & (df["Data"].str.len() < 100)]
df = df.drop_duplicates(subset="Data").reset_index(drop=True)
df["Label"] = df["Label"].astype(int)
print("after filtering:", len(df), dict(df["Label"].value_counts()))

if TOTAL and len(df) > TOTAL:
    df = df.groupby("Label", group_keys=False).apply(
        lambda d: d.sample(frac=TOTAL / len(df), random_state=SEED)).reset_index(drop=True)

# stratified 80 / 10 / 10
train_raw, rest = train_test_split(df, test_size=0.20, stratify=df["Label"], random_state=SEED)
val_raw, test_raw = train_test_split(rest, test_size=0.50, stratify=rest["Label"], random_state=SEED)
train_raw, val_raw, test_raw = [d.reset_index(drop=True) for d in (train_raw, val_raw, test_raw)]

print(train_raw.shape, val_raw.shape, test_raw.shape)
print("label mix (train/val/test):",
      [dict(d["Label"].value_counts().sort_index()) for d in (train_raw, val_raw, test_raw)])# ===== DROP-IN REPLACEMENT for the BASE_URL / read_csv cell =====
# Produces train_raw, val_raw, test_raw with columns "Data" and "Label" (0=neutral, 1=positive, 2=negative),
# with the column names the rest of the notebook expects.
!pip install -q datasets
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 17
TOTAL = 15700          # subsample size (about 15.7k rows); set None to use everything

ds = load_dataset("SayedShaun/sentigold")
raw = ds["train"].to_pandas()
print("columns:", raw.columns.tolist())
print(raw["Polarity"].value_counts())

# 5 -> 3 classes, same ids as the notebook's LABEL_MAP (neutral=0, positive=1, negative=2)
COLLAPSE = {"SP": 1, "WP": 1, "NU": 0, "WN": 2, "SN": 2}
unknown = set(raw["Polarity"].unique()) - set(COLLAPSE)
assert not unknown, f"Unexpected Polarity values {unknown}: edit COLLAPSE using the counts printed above"

df = pd.DataFrame({"Data": raw["Text"], "Label": raw["Polarity"].map(COLLAPSE)})
if "Domain" in raw.columns:
    df["Domain"] = raw["Domain"]          # extra column, ignored by clean(); handy for later analysis

# pre-apply the notebook's own filters (non-null, 1 < length < 100, no duplicates) so clean() doesn't shrink it much
df = df.dropna(subset=["Data", "Label"])
df["Data"] = df["Data"].astype(str).str.strip()
df = df[(df["Data"].str.len() > 1) & (df["Data"].str.len() < 100)]
df = df.drop_duplicates(subset="Data").reset_index(drop=True)
df["Label"] = df["Label"].astype(int)
print("after filtering:", len(df), dict(df["Label"].value_counts()))

if TOTAL and len(df) > TOTAL:
    df = df.groupby("Label", group_keys=False).apply(
        lambda d: d.sample(frac=TOTAL / len(df), random_state=SEED)).reset_index(drop=True)

# stratified 80 / 10 / 10
train_raw, rest = train_test_split(df, test_size=0.20, stratify=df["Label"], random_state=SEED)
val_raw, test_raw = train_test_split(rest, test_size=0.50, stratify=rest["Label"], random_state=SEED)
train_raw, val_raw, test_raw = [d.reset_index(drop=True) for d in (train_raw, val_raw, test_raw)]

print(train_raw.shape, val_raw.shape, test_raw.shape)
print("label mix (train/val/test):",
      [dict(d["Label"].value_counts().sort_index()) for d in (train_raw, val_raw, test_raw)])

README.md:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.42MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/70000 [00:00<?, ? examples/s]

columns: ['ID', 'Text', 'Polarity', 'Domain']
Polarity
SP    15318
NU    15156
SN    13767
WN    13682
WP    12077
Name: count, dtype: int64
after filtering: 65257 {2: np.int64(25555), 1: np.int64(25298), 0: np.int64(14404)}
(12559, 3) (1570, 3) (1570, 3)
label mix (train/val/test): [{0: np.int64(2772), 1: np.int64(4869), 2: np.int64(4918)}, {0: np.int64(347), 1: np.int64(608), 2: np.int64(615)}, {0: np.int64(346), 1: np.int64(609), 2: np.int64(615)}]


/tmp/ipykernel_1273/1101286140.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("Label", group_keys=False).apply(


columns: ['ID', 'Text', 'Polarity', 'Domain']
Polarity
SP    15318
NU    15156
SN    13767
WN    13682
WP    12077
Name: count, dtype: int64
after filtering: 65257 {2: np.int64(25555), 1: np.int64(25298), 0: np.int64(14404)}
(12559, 3) (1570, 3) (1570, 3)
label mix (train/val/test): [{0: np.int64(2772), 1: np.int64(4869), 2: np.int64(4918)}, {0: np.int64(347), 1: np.int64(608), 2: np.int64(615)}, {0: np.int64(346), 1: np.int64(609), 2: np.int64(615)}]


/tmp/ipykernel_1273/1101286140.py:79: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("Label", group_keys=False).apply(


In [ ]:
def clean(df):
    df = df.rename(columns={"Data": "sentence", "Label": "sentiment"}).copy()
    df = df.dropna(subset=["sentence", "sentiment"])
    df["sentence"] = df["sentence"].astype(str).str.strip()
    df = df[(df["sentence"].str.len() > 1) & (df["sentence"].str.len() < 100)]
    df = df.drop_duplicates(subset="sentence").reset_index(drop=True)
    return df[["sentence", "sentiment"]]

pool_source = clean(pd.concat([train_raw, val_raw], ignore_index=True))
test_df     = clean(test_raw)

pool_source = pool_source[~pool_source["sentence"].isin(set(test_df["sentence"]))].reset_index(drop=True)

print("Pool source:", len(pool_source))
print("Locked test:", len(test_df))

Pool source: 14129
Locked test: 1570


In [ ]:
def make_split(pool_source, seed_size=200, seed=17):
    frac = seed_size / len(pool_source)
    seed_train = pool_source.groupby("sentiment", group_keys=False).sample(frac=frac, random_state=seed)
    pool_df = pool_source.drop(seed_train.index).reset_index(drop=True)
    return seed_train.reset_index(drop=True), pool_df

seed_train, pool_df = make_split(pool_source, seed_size=200, seed=17)
print("Seed train:", len(seed_train), dict(seed_train["sentiment"].value_counts()))
print("Pool:", len(pool_df))

Seed train: 200 {1: np.int64(78), 2: np.int64(78), 0: np.int64(44)}
Pool: 13929


In [ ]:
def least_confidence(probs):
    p_max = np.max(probs)
    n = len(probs)
    return -(1 - p_max) * (n / (n - 1))

def margin_sampling(probs):
    top2 = np.sort(probs)[::-1][:2]
    return top2[0] - top2[1]

def ratio_sampling(probs):
    top2 = np.sort(probs)[::-1][:2]
    return top2[0] / (top2[1] + 1e-12)

def entropy_sampling(probs):
    p = np.clip(probs, 1e-12, 1)
    ent = -np.sum(p * np.log2(p)) / np.log2(len(probs))
    return -ent

def random_sampling(probs, rng):
    return rng.random()

STRATEGIES = {
    "least_confidence": least_confidence,
    "margin": margin_sampling,
    "ratio": ratio_sampling,
    "entropy": entropy_sampling,
    "random": random_sampling,
}

In [ ]:
import pickle
with open(f"{CKPT_DIR}/sentigold_human_annotator_margin.pkl", "rb") as f:
    st = pickle.load(f)
first = st["train_df"].iloc[:len(seed_train)]
bench = pd.read_csv(f"{CKPT_DIR}/annotator_benchmark.csv")
print("seed rows here:", len(seed_train), dict(seed_train["sentiment"].value_counts().sort_index()))
print("pool rows here:", len(pool_df))
print("seed == first rows of the AL runs' training set:",
      set(zip(first["sentence"], first["sentiment"])) == set(zip(seed_train["sentence"], seed_train["sentiment"])))
print("benchmark rows that are also seed rows (want 0):", len(set(bench["sentence"]) & set(seed_train["sentence"])))
print("all benchmark rows come from the pool (want True):", bool(bench["sentence"].isin(pool_df["sentence"]).all()))

seed rows here: 200 {0: np.int64(44), 1: np.int64(78), 2: np.int64(78)}
pool rows here: 13929
seed == first rows of the AL runs' training set: True
benchmark rows that are also seed rows (want 0): 0
all benchmark rows come from the pool (want True): True


In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

llm_tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/qwen2.5-7b-instruct")
llm_model = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/qwen2.5-7b-instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
LABEL_MAP = {"neutral": 0, "positive": 1, "negative": 2}
LABEL_NAMES = {v: k for k, v in LABEL_MAP.items()}

def build_prompt(sentence, examples=None):
    instructions = (
        "You are a sentiment classifier for Bangla text. "
        "Classify the sentiment of the given text as exactly one of: positive, negative, neutral. "
        "Respond with only the single label word, nothing else."
    )
    few_shot_block = ""
    if examples:
        for ex_text, ex_label in examples:
            few_shot_block += f"\nText: {ex_text}\nLabel: {LABEL_NAMES[ex_label]}"
    return f"{instructions}\n{few_shot_block}\n\nText: {sentence}\nLabel:"

In [ ]:
def make_chat_text(sentence, examples=None):
    msgs = [{"role": "user", "content": build_prompt(sentence, examples)}]
    return llm_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

llm_tokenizer.padding_side = "left"
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

LABEL_WORDS = [LABEL_NAMES[i] for i in range(3)]
LABEL_TOKEN_IDS = [llm_tokenizer.encode(w, add_special_tokens=False)[0] for w in LABEL_WORDS]
assert len(set(LABEL_TOKEN_IDS)) == 3, "label first-tokens collide; use a different label word"

In [ ]:
import numpy as np, pandas as pd, torch

llm_tokenizer.padding_side = "left"      # needed for batching with a decoder-only model

@torch.no_grad()
def llm_logprobs_rows(sentences, batch_size=16):
    """(n,3) log-scores over [neutral, positive, negative]. Same prompt/model as before, nothing else changed."""
    texts = [make_chat_text(s) for s in sentences]
    out = np.zeros((len(texts), 3), dtype=np.float32)
    for i in range(0, len(texts), batch_size):
        enc = llm_tokenizer(texts[i:i+batch_size], return_tensors="pt", padding=True).to(llm_model.device)
        logits = llm_model(**enc).logits[:, -1, :]
        out[i:i+batch_size] = torch.log_softmax(logits[:, LABEL_TOKEN_IDS].float(), dim=-1).cpu().numpy()
        if (i // batch_size) % 20 == 0:
            print(f"{min(i+batch_size, len(texts))}/{len(texts)}")
    return out

bench = pd.read_csv(f"{CKPT_DIR}/annotator_benchmark.csv")
LP_seed  = llm_logprobs_rows(seed_train["sentence"].tolist())
LP_bench = llm_logprobs_rows(bench["sentence"].tolist())

np.savez(f"{CKPT_DIR}/step2_logprobs.npz",
         LP_seed=LP_seed, y_seed=seed_train["sentiment"].values,
         LP_bench=LP_bench, y_bench=bench["gold"].values,
         source_bench=bench["source"].values)

# sanity check: does this run reproduce the labels logged during the AL runs?
same = (LP_bench.argmax(1) == bench["zs_label"].values).mean()
print(f"argmax agrees with the logged Step-1 labels on {same:.1%} of benchmark rows")
print("accuracy on benchmark:", (LP_bench.argmax(1) == bench['gold'].values).mean().round(3), "(Step 1 said 0.617)")
print("accuracy on the 200 seed rows:", (LP_seed.argmax(1) == seed_train['sentiment'].values).mean().round(3))

16/200


KeyboardInterrupt: 

In [ ]:
import json
import numpy as np
from scipy.optimize import minimize
from scipy.special import log_softmax
from sklearn.model_selection import StratifiedKFold

d = np.load(f"{CKPT_DIR}/step2_logprobs.npz", allow_pickle=True)
LP, y = d["LP_seed"], d["y_seed"].astype(int)          # columns: 0=neutral, 1=positive, 2=negative

def shift(LP, b):
    """b = (neutral_offset, negative_offset); positive is fixed at 0. Returns re-normalised log-scores."""
    return log_softmax(LP + np.array([b[0], 0.0, b[1]]), axis=1)

def ce(b, LP, y):
    return -shift(LP, b)[np.arange(len(y)), y].mean()

def fit(LP, y):
    return minimize(ce, x0=np.zeros(2), args=(LP, y), method="Nelder-Mead").x

# ---- 1. fit on all 200 seed rows ----
b = fit(LP, y)
acc = lambda LPx, yx, bx: (shift(LPx, bx).argmax(1) == yx).mean()
print(f"offsets: neutral {b[0]:+.2f} | positive +0.00 (fixed) | negative {b[1]:+.2f}")
print(f"cross-entropy on seed: {ce(np.zeros(2), LP, y):.3f} -> {ce(b, LP, y):.3f}")
print(f"accuracy on seed (in-sample, optimistic): {acc(LP, y, np.zeros(2)):.3f} -> {acc(LP, y, b):.3f}")
pred0, pred1 = LP.argmax(1), shift(LP, b).argmax(1)
print("share predicted neutral/positive/negative")
print("  gold   :", np.round(np.bincount(y, minlength=3) / len(y), 3))
print("  before :", np.round(np.bincount(pred0, minlength=3) / len(y), 3))
print("  after  :", np.round(np.bincount(pred1, minlength=3) / len(y), 3))

# ---- 2. honest accuracy inside the seed set: fit on 4/5, score on the held-out 1/5, repeat ----
skf = StratifiedKFold(5, shuffle=True, random_state=17)
hit = np.zeros(len(y), bool)
for tr, te in skf.split(LP, y):
    hit[te] = shift(LP[te], fit(LP[tr], y[tr])).argmax(1) == y[te]
print(f"5-fold cross-validated accuracy on seed: {hit.mean():.3f}  (no calibration: {(pred0 == y).mean():.3f})")

# ---- 3. how stable are the offsets? refit on 200 bootstrap resamples of the seed rows ----
rng = np.random.default_rng(17)
B = np.array([fit(LP[i], y[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(200))])
lo, hi = np.percentile(B, [2.5, 97.5], axis=0)
print(f"bootstrap 95% range: neutral [{lo[0]:+.2f}, {hi[0]:+.2f}] | negative [{lo[1]:+.2f}, {hi[1]:+.2f}]")

json.dump({"neutral": float(b[0]), "positive": 0.0, "negative": float(b[1])},
          open(f"{CKPT_DIR}/step2_offsets.json", "w"))
print("saved step2_offsets.json")

offsets: neutral -6.01 | positive +0.00 (fixed) | negative +0.39
cross-entropy on seed: 2.377 -> 1.737
accuracy on seed (in-sample, optimistic): 0.650 -> 0.745
share predicted neutral/positive/negative
  gold   : [0.22 0.39 0.39]
  before : [0.415 0.3   0.285]
  after  : [0.205 0.395 0.4  ]
5-fold cross-validated accuracy on seed: 0.730  (no calibration: 0.650)
bootstrap 95% range: neutral [-8.13, -3.66] | negative [-3.28, +3.95]
saved step2_offsets.json


In [ ]:
import json
import numpy as np, pandas as pd
from scipy.optimize import minimize
from scipy.special import log_softmax
from scipy.stats import binomtest
from sklearn.metrics import f1_score

d = np.load(f"{CKPT_DIR}/step2_logprobs.npz", allow_pickle=True)
LPs, ys = d["LP_seed"], d["y_seed"].astype(int)
LPb, yb, src = d["LP_bench"], d["y_bench"].astype(int), d["source_bench"].astype(str)
o = json.load(open(f"{CKPT_DIR}/step2_offsets.json"))
b2 = np.array([o["neutral"], o["negative"]])                      # primary: both offsets, as fitted in 2c

def shift(LP, b):
    return log_softmax(LP + np.array([b[0], 0.0, b[1]]), axis=1)

# secondary (decided before looking at the benchmark): neutral-only offset, negative fixed at 0, fitted on seed only
ce1 = lambda t: -shift(LPs, [t[0], 0.0])[np.arange(len(ys)), ys].mean()
b1 = np.array([minimize(ce1, x0=[0.0], method="Nelder-Mead").x[0], 0.0])
print(f"neutral-only offset (fitted on seed): {b1[0]:+.2f}\n")

variants = {"baseline": np.zeros(2), "2 offsets": b2, "neutral only": b1}
preds = {k: shift(LPb, v).argmax(1) for k, v in variants.items()}

rng = np.random.default_rng(17)
def report(mask, name):
    n = mask.sum(); base = preds["baseline"][mask] == yb[mask]
    print(f"{name:8s} n={n}")
    for k in variants:
        p = preds[k][mask]; ok = p == yb[mask]
        line = (f"   {k:13s} acc={ok.mean():.3f} macroF1={f1_score(yb[mask], p, average='macro'):.3f} "
                f"says-neutral={np.mean(p == 0):.0%} (gold {np.mean(yb[mask] == 0):.0%})")
        if k != "baseline":
            diff = ok.astype(int) - base.astype(int)
            bs = [diff[rng.integers(0, n, n)].mean() for _ in range(2000)]
            lo, hi = np.percentile(bs, [2.5, 97.5])
            w, l = int(((diff == 1)).sum()), int(((diff == -1)).sum())     # fixed vs broken rows
            pval = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
            line += f" | change {diff.mean():+.3f} (95% CI {lo:+.3f}..{hi:+.3f}) fixed {w}, broke {l}, McNemar p={pval:.3g}"
        print(line)
    print()

report(np.ones(len(yb), bool), "ALL")
for s in ["random", "margin", "entropy"]:
    report(src == s, s)

out = pd.DataFrame({"source": src, "gold": yb, **{f"pred_{k.replace(' ', '_')}": v for k, v in preds.items()}})
out.to_csv(f"{CKPT_DIR}/step2d_predictions.csv", index=False)
print("saved step2d_predictions.csv")

neutral-only offset (fitted on seed): -6.17

ALL      n=1707
   baseline      acc=0.619 macroF1=0.617 says-neutral=42% (gold 23%)
   2 offsets     acc=0.670 macroF1=0.638 says-neutral=17% (gold 23%) | change +0.052 (95% CI +0.029..+0.073) fixed 220, broke 132, McNemar p=3.16e-06
   neutral only  acc=0.667 macroF1=0.635 says-neutral=17% (gold 23%) | change +0.048 (95% CI +0.027..+0.069) fixed 214, broke 132, McNemar p=1.22e-05

random   n=600
   baseline      acc=0.708 macroF1=0.709 says-neutral=41% (gold 26%)
   2 offsets     acc=0.737 macroF1=0.717 says-neutral=20% (gold 26%) | change +0.028 (95% CI -0.005..+0.062) fixed 63, broke 46, McNemar p=0.125
   neutral only  acc=0.733 macroF1=0.714 says-neutral=20% (gold 26%) | change +0.025 (95% CI -0.008..+0.057) fixed 61, broke 46, McNemar p=0.176

margin   n=563
   baseline      acc=0.574 macroF1=0.573 says-neutral=45% (gold 23%)
   2 offsets     acc=0.627 macroF1=0.592 says-neutral=18% (gold 23%) | change +0.053 (95% CI +0.014..+0.092) f

In [ ]:
import os
import numpy as np, pandas as pd, torch

DRAW_SEED, K = 17, 4                       # K examples per class, drawn from the seed rows
TAG = f"fs_k{K}_s{DRAW_SEED}"

def pick_shots(k_per_class=4, seed=17):    # identical to the 3a notebook, so the prompt matches that run when seed=17
    shots = []
    for lab in range(3):
        rows = seed_train[seed_train.sentiment == lab].sample(k_per_class, random_state=seed)
        shots += list(zip(rows.sentence, rows.sentiment))
    rng = np.random.default_rng(seed); rng.shuffle(shots)
    return shots

shots = pick_shots(K, DRAW_SEED)
shot_sents = {s for s, _ in shots}
print(f"{len(shots)} examples in the prompt:", [LABEL_NAMES[l] for _, l in shots])
print("tokens in one full prompt:", len(llm_tokenizer(make_chat_text('x', shots))['input_ids']))

llm_tokenizer.padding_side = "left"

@torch.no_grad()
def logprobs_chunked(sentences, examples, name, batch_size=8, chunk=256):
    """Log-scores (n,3). Every finished chunk of 256 rows is saved to Drive, so a disconnect loses at most one chunk."""
    parts = []
    for j in range(0, len(sentences), chunk):
        path = f"{CKPT_DIR}/step3_{TAG}_{name}_{j // chunk:03d}.npy"
        if os.path.exists(path):
            parts.append(np.load(path)); continue
        texts = [make_chat_text(s, examples) for s in sentences[j:j + chunk]]
        out = np.zeros((len(texts), 3), dtype=np.float32)
        for i in range(0, len(texts), batch_size):
            enc = llm_tokenizer(texts[i:i + batch_size], return_tensors="pt", padding=True).to(llm_model.device)
            logits = llm_model(**enc).logits[:, -1, :]
            out[i:i + batch_size] = torch.log_softmax(logits[:, LABEL_TOKEN_IDS].float(), dim=-1).cpu().numpy()
        np.save(path, out); parts.append(out)
        print(f"  {name}: {min(j + chunk, len(sentences))}/{len(sentences)}")
    return np.concatenate(parts)

bench = pd.read_csv(f"{CKPT_DIR}/annotator_benchmark.csv")
keep = ~seed_train["sentence"].isin(shot_sents).values            # seed rows NOT shown in the prompt
LPs_all = logprobs_chunked(seed_train["sentence"].tolist(), shots, "seed")
LPb     = logprobs_chunked(bench["sentence"].tolist(), shots, "bench")
np.savez(f"{CKPT_DIR}/step3_{TAG}.npz", LP_seed=LPs_all, keep_seed=keep, LP_bench=LPb)
print("saved. seed rows usable for fitting (not in the prompt):", int(keep.sum()))
print("raw few-shot accuracy on those seed rows:", (LPs_all[keep].argmax(1) == seed_train["sentiment"].values[keep]).mean().round(3))
print("raw few-shot accuracy on benchmark:", (LPb.argmax(1) == bench["gold"].values).mean().round(3), "(zero-shot was 0.619)")

In [ ]:
import numpy as np, pandas as pd
from scipy.optimize import minimize
from scipy.special import log_softmax
from scipy.stats import binomtest
from sklearn.metrics import f1_score

TAG = "fs_k4_s17"
z = np.load(f"{CKPT_DIR}/step2_logprobs.npz", allow_pickle=True)
f = np.load(f"{CKPT_DIR}/step3_{TAG}.npz", allow_pickle=True)
keep = f["keep_seed"]
ys = z["y_seed"].astype(int)[keep]
yb, src = z["y_bench"].astype(int), z["source_bench"].astype(str)
ZS_s, ZS_b = z["LP_seed"][keep], z["LP_bench"]
FS_s, FS_b = f["LP_seed"][keep], f["LP_bench"]

def shift(LP, b): return log_softmax(LP + np.array([b[0], 0.0, b[1]]), axis=1)
def fit(LP, y):
    ce = lambda b: -shift(LP, b)[np.arange(len(y)), y].mean()
    return minimize(ce, x0=np.zeros(2), method="Nelder-Mead").x

b_zs, b_fs = fit(ZS_s, ys), fit(FS_s, ys)          # both fitted on the SAME seed rows (the ones not in the few-shot prompt)
print(f"fit on {len(ys)} seed rows | zero-shot offsets {np.round(b_zs, 2)} | few-shot offsets {np.round(b_fs, 2)}\n")

preds = {"zero-shot":       shift(ZS_b, [0, 0]).argmax(1),
         "zero-shot + cal": shift(ZS_b, b_zs).argmax(1),
         "few-shot":        shift(FS_b, [0, 0]).argmax(1),
         "few-shot + cal":  shift(FS_b, b_fs).argmax(1)}

rng = np.random.default_rng(17)
def compare(a, b, mask):
    """accuracy change b-vs-a on the rows in mask, with bootstrap CI and McNemar p."""
    oa, ob = preds[a][mask] == yb[mask], preds[b][mask] == yb[mask]
    diff = ob.astype(int) - oa.astype(int); n = len(diff)
    lo, hi = np.percentile([diff[rng.integers(0, n, n)].mean() for _ in range(2000)], [2.5, 97.5])
    w, l = int((diff == 1).sum()), int((diff == -1).sum())
    p = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
    return f"{diff.mean():+.3f} (CI {lo:+.3f}..{hi:+.3f}) fixed {w} broke {l} p={p:.3g}"

def report(mask, name):
    print(f"{name} n={mask.sum()}   (gold neutral {np.mean(yb[mask] == 0):.0%})")
    for k, p in preds.items():
        pm = p[mask]
        print(f"   {k:16s} acc={np.mean(pm == yb[mask]):.3f} macroF1={f1_score(yb[mask], pm, average='macro'):.3f} says-neutral={np.mean(pm == 0):.0%}")
    print("   few-shot vs zero-shot          :", compare("zero-shot", "few-shot", mask))
    print("   few-shot+cal vs zero-shot+cal  :", compare("zero-shot + cal", "few-shot + cal", mask))
    print()

report(np.ones(len(yb), bool), "ALL")
for s in ["random", "margin", "entropy"]:
    report(src == s, s)
pd.DataFrame({"source": src, "gold": yb, **{f"pred_{k.replace(' ', '_').replace('+', 'plus')}": v for k, v in preds.items()}}
             ).to_csv(f"{CKPT_DIR}/step3_{TAG}_predictions.csv", index=False)

fit on 188 seed rows | zero-shot offsets [-5.94  0.63] | few-shot offsets [-3.1  -2.94]

ALL n=1707   (gold neutral 23%)
   zero-shot        acc=0.619 macroF1=0.617 says-neutral=42%
   zero-shot + cal  acc=0.672 macroF1=0.640 says-neutral=17%
   few-shot         acc=0.678 macroF1=0.660 says-neutral=26%
   few-shot + cal   acc=0.694 macroF1=0.667 says-neutral=18%
   few-shot vs zero-shot          : +0.059 (CI +0.039..+0.080) fixed 212 broke 111 p=2.04e-08
   few-shot+cal vs zero-shot+cal  : +0.022 (CI +0.005..+0.039) fixed 136 broke 98 p=0.0154

random n=600   (gold neutral 26%)
   zero-shot        acc=0.708 macroF1=0.709 says-neutral=41%
   zero-shot + cal  acc=0.740 macroF1=0.720 says-neutral=20%
   few-shot         acc=0.750 macroF1=0.742 says-neutral=28%
   few-shot + cal   acc=0.752 macroF1=0.736 says-neutral=21%
   few-shot vs zero-shot          : +0.042 (CI +0.012..+0.073) fixed 60 broke 35 p=0.0134
   few-shot+cal vs zero-shot+cal  : +0.012 (CI -0.017..+0.040) fixed 39 broke 32 

In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import log_softmax
from sklearn.metrics import f1_score

DRAWS = [17, 18, 19]
K = 4
z = np.load(f"{CKPT_DIR}/step2_logprobs.npz", allow_pickle=True)
yb, src = z["y_bench"].astype(int), z["source_bench"].astype(str)

def shift(LP, b): return log_softmax(LP + np.array([b[0], 0.0, b[1]]), axis=1)
def fit(LP, y):
    ce = lambda b: -shift(LP, b)[np.arange(len(y)), y].mean()
    return minimize(ce, x0=np.zeros(2), method="Nelder-Mead").x

groups = {"ALL": np.ones(len(yb), bool), **{s: src == s for s in ["random", "margin", "entropy"]}}
names = ["zero-shot", "zero-shot + cal", "few-shot", "few-shot + cal"]
acc, f1, neu = {}, {}, {}                      # [variant][group] -> list over draws
for dr in DRAWS:
    f = np.load(f"{CKPT_DIR}/step3_fs_k{K}_s{dr}.npz", allow_pickle=True)
    keep = f["keep_seed"]; ys = z["y_seed"].astype(int)[keep]
    ZS_s, ZS_b, FS_s, FS_b = z["LP_seed"][keep], z["LP_bench"], f["LP_seed"][keep], f["LP_bench"]
    b_zs, b_fs = fit(ZS_s, ys), fit(FS_s, ys)
    P = {"zero-shot": ZS_b.argmax(1), "zero-shot + cal": shift(ZS_b, b_zs).argmax(1),
         "few-shot": FS_b.argmax(1), "few-shot + cal": shift(FS_b, b_fs).argmax(1)}
    print(f"draw {dr}: fitted on {len(ys)} seed rows | few-shot offsets {np.round(b_fs, 2)}")
    for n in names:
        for g, m in groups.items():
            acc.setdefault(n, {}).setdefault(g, []).append(np.mean(P[n][m] == yb[m]))
            f1.setdefault(n, {}).setdefault(g, []).append(f1_score(yb[m], P[n][m], average="macro"))
            neu.setdefault(n, {}).setdefault(g, []).append(np.mean(P[n][m] == 0))

def table(title, D, fmt="{:.3f}"):
    print(f"\n{title}   (columns: one per draw {DRAWS} | mean | min..max)")
    for g in groups:
        print(f"  {g} (gold neutral {np.mean(yb[groups[g]] == 0):.0%})" if title.startswith("says") else f"  {g}")
        for n in names:
            v = np.array(D[n][g])
            print(f"    {n:16s}", " ".join(fmt.format(x) for x in v), f"| {fmt.format(v.mean())} | {fmt.format(v.min())}..{fmt.format(v.max())}")
table("ACCURACY", acc); table("MACRO-F1", f1); table("says-neutral share", neu, "{:.2f}")

print("\nGAIN of few-shot + cal over zero-shot + cal (accuracy points), per draw")
for g in groups:
    d = np.array(acc["few-shot + cal"][g]) - np.array(acc["zero-shot + cal"][g])
    print(f"  {g:8s}", " ".join(f"{x:+.3f}" for x in d), f"| mean {d.mean():+.3f}")

draw 17: fitted on 188 seed rows | few-shot offsets [-3.1  -2.94]
draw 18: fitted on 188 seed rows | few-shot offsets [-1.72 -3.7 ]
draw 19: fitted on 188 seed rows | few-shot offsets [-3.04 -4.21]

ACCURACY   (columns: one per draw [17, 18, 19] | mean | min..max)
  ALL
    zero-shot        0.619 0.619 0.619 | 0.619 | 0.619..0.619
    zero-shot + cal  0.672 0.670 0.672 | 0.671 | 0.670..0.672
    few-shot         0.678 0.683 0.671 | 0.677 | 0.671..0.683
    few-shot + cal   0.694 0.686 0.678 | 0.686 | 0.678..0.694
  random
    zero-shot        0.708 0.708 0.708 | 0.708 | 0.708..0.708
    zero-shot + cal  0.740 0.737 0.740 | 0.739 | 0.737..0.740
    few-shot         0.750 0.742 0.752 | 0.748 | 0.742..0.752
    few-shot + cal   0.752 0.743 0.750 | 0.748 | 0.743..0.752
  margin
    zero-shot        0.574 0.574 0.574 | 0.574 | 0.574..0.574
    zero-shot + cal  0.631 0.629 0.631 | 0.630 | 0.629..0.631
    few-shot         0.648 0.648 0.638 | 0.645 | 0.638..0.648
    few-shot + cal   0.666 0.

In [ ]:
!pip install -q sentence-transformers

import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

bench = pd.read_csv(f"{CKPT_DIR}/annotator_benchmark.csv")
S_text, S_y = seed_train["sentence"].tolist(), seed_train["sentiment"].values      # the knowledge base: 200 gold seed rows
Q_text, Q_y, Q_src = bench["sentence"].tolist(), bench["gold"].values, bench["source"].values
assert not set(S_text) & set(Q_text), "benchmark rows must not be in the knowledge base"

def tfidf_sims():
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True).fit(S_text + Q_text)
    return (v.transform(Q_text) @ v.transform(S_text).T).toarray()                # cosine (rows are L2-normalised)

def dense_sims(name, prefix=""):
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(name)
    enc = lambda t: m.encode([prefix + s for s in t], batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    return enc(Q_text) @ enc(S_text).T

def evaluate(name, sims, ks=(1, 5, 12)):
    order = np.argsort(-sims, axis=1)
    chance = np.array([(S_y == c).mean() for c in Q_y]).mean()                   # agreement expected from random neighbours
    print(f"\n== {name} ==  (random-neighbour agreement would be {chance:.3f})")
    if "random" not in name:
        print("  nearest-neighbour similarity: median %.2f | 99th percentile %.2f" % (np.median(sims.max(1)), np.percentile(sims.max(1), 99)))
    for k in ks:
        nb = order[:, :k]
        agree = (S_y[nb] == Q_y[:, None]).mean(1)                                 # share of the k neighbours with the query's gold label
        votes = np.zeros((len(Q_y), 3))
        for c in range(3):
            votes[:, c] = ((S_y[nb] == c) * np.take_along_axis(sims, nb, 1)).sum(1)   # similarity-weighted vote
        acc = (votes.argmax(1) == Q_y).mean()
        per = {s: round(float(agree[Q_src == s].mean()), 3) for s in ("random", "margin", "entropy")}
        print(f"  k={k:2d} neighbour-label agreement {agree.mean():.3f} (by source {per}) | kNN vote accuracy {acc:.3f}")

rng = np.random.default_rng(17)
evaluate("random similarity (sanity: should look like chance)", rng.random((len(Q_text), len(S_text))))
evaluate("TF-IDF character n-grams (word-overlap baseline, no model)", tfidf_sims())
print("\nalways predicting the single most common benchmark class would score", max((Q_y == c).mean() for c in range(3)).round(3))


== random similarity (sanity: should look like chance) ==  (random-neighbour agreement would be 0.352)
  k= 1 neighbour-label agreement 0.344 (by source {'random': 0.33, 'margin': 0.359, 'entropy': 0.346}) | kNN vote accuracy 0.344
  k= 5 neighbour-label agreement 0.353 (by source {'random': 0.348, 'margin': 0.35, 'entropy': 0.362}) | kNN vote accuracy 0.361
  k=12 neighbour-label agreement 0.347 (by source {'random': 0.346, 'margin': 0.344, 'entropy': 0.352}) | kNN vote accuracy 0.360

== TF-IDF character n-grams (word-overlap baseline, no model) ==  (random-neighbour agreement would be 0.352)
  nearest-neighbour similarity: median 0.16 | 99th percentile 0.33
  k= 1 neighbour-label agreement 0.429 (by source {'random': 0.438, 'margin': 0.416, 'entropy': 0.432}) | kNN vote accuracy 0.429
  k= 5 neighbour-label agreement 0.398 (by source {'random': 0.408, 'margin': 0.39, 'entropy': 0.395}) | kNN vote accuracy 0.436
  k=12 neighbour-label agreement 0.386 (by source {'random': 0.387, 'ma

In [ ]:
evaluate("multilingual-e5-large", dense_sims("intfloat/multilingual-e5-large", prefix="query: "))
evaluate("BGE-M3", dense_sims("BAAI/bge-m3"))

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

... (21 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


== BGE-M3 ==  (random-neighbour agreement would be 0.352)
  nearest-neighbour similarity: median 0.57 | 99th percentile 0.71
  k= 1 neighbour-label agreement 0.504 (by source {'random': 0.528, 'margin': 0.506, 'entropy': 0.476}) | kNN vote accuracy 0.504
  k= 5 neighbour-label agreement 0.497 (by source {'random': 0.513, 'margin': 0.496, 'entropy': 0.481}) | kNN vote accuracy 0.566
  k=12 neighbour-label agreement 0.487 (by source {'random': 0.507, 'margin': 0.48, 'entropy': 0.472}) | kNN vote accuracy 0.586


In [ ]:
!pip install -q sentence-transformers

import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

bench = pd.read_csv(f"{CKPT_DIR}/annotator_benchmark.csv")
S_text = seed_train["sentence"].tolist()
Q_text = bench["sentence"].tolist()
assert not set(S_text) & set(Q_text), "benchmark rows must not be in the knowledge base"

m = SentenceTransformer("BAAI/bge-m3")
enc = lambda t: m.encode(t, batch_size=64, normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
E_seed, E_bench = enc(S_text), enc(Q_text)
np.savez(f"{CKPT_DIR}/step4_embeddings.npz", E_seed=E_seed, E_bench=E_bench)
print("saved vectors:", E_seed.shape, E_bench.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

saved vectors: (200, 1024) (1707, 1024)


In [ ]:
import numpy as np, pandas as pd

bench = pd.read_csv(f"{CKPT_DIR}/annotator_benchmark.csv")
E = np.load(f"{CKPT_DIR}/step4_embeddings.npz")
E_seed, E_bench = E["E_seed"], E["E_bench"]
S_y = seed_train["sentiment"].values
Q_y, Q_src = bench["gold"].values, bench["source"].values
KMAX = 12                                   # the largest k we will try; smaller k uses the first k columns

# --- retrieved neighbours: most similar first. Seed rows may not retrieve themselves (leave-one-out) ---
sim_ss = E_seed @ E_seed.T
np.fill_diagonal(sim_ss, -np.inf)
NB_seed = np.argsort(-sim_ss, axis=1)[:, :KMAX]
NB_bench = np.argsort(-(E_bench @ E_seed.T), axis=1)[:, :KMAX]
assert not (NB_seed == np.arange(len(S_y))[:, None]).any(), "a seed row retrieved itself"

# --- random-control neighbours: same shape, chosen without looking at similarity (fixed seed, also leave-one-out) ---
rng = np.random.default_rng(17)
RND_bench = np.array([rng.permutation(len(S_y))[:KMAX] for _ in range(len(Q_y))])
RND_seed = np.array([rng.permutation(np.delete(np.arange(len(S_y)), i))[:KMAX] for i in range(len(S_y))])
assert not (RND_seed == np.arange(len(S_y))[:, None]).any()
np.savez(f"{CKPT_DIR}/step4_neighbours.npz", NB_seed=NB_seed, NB_bench=NB_bench, RND_seed=RND_seed, RND_bench=RND_bench)
print("saved neighbour lists:", NB_seed.shape, NB_bench.shape)

# --- does the label mix of retrieved examples look sensible? (counts over all queries, k=8) ---
for nm, nb in [("retrieved", NB_bench[:, :8]), ("random", RND_bench[:, :8])]:
    print(f"  {nm:9s} examples shown, label shares neutral/positive/negative:", np.round(np.bincount(S_y[nb].ravel(), minlength=3) / nb.size, 3))

# --- how much does knowledge-base size matter? agreement of top-k neighbours' labels with the query's gold label ---
chance = np.array([(S_y == c).mean() for c in Q_y]).mean()
print(f"\nneighbour-label agreement vs knowledge-base size (random subsets of the 200 seeds; chance = {chance:.3f})")
rng = np.random.default_rng(17)
for size in (25, 50, 100, 200):
    res = {1: [], 5: []}
    for _ in range(1 if size == 200 else 30):
        idx = rng.choice(len(S_y), size, replace=False)
        order = np.argsort(-(E_bench @ E_seed[idx].T), axis=1)
        for k in res:
            res[k].append((S_y[idx][order[:, :k]] == Q_y[:, None]).mean())
    print(f"  {size:3d} seed rows | k=1: {np.mean(res[1]):.3f} | k=5: {np.mean(res[5]):.3f}" + (f"  (spread over 30 subsets: k=5 {np.min(res[5]):.3f}..{np.max(res[5]):.3f})" if size < 200 else ""))

saved neighbour lists: (200, 12) (1707, 12)
  retrieved examples shown, label shares neutral/positive/negative: [0.147 0.438 0.415]
  random    examples shown, label shares neutral/positive/negative: [0.217 0.391 0.392]

neighbour-label agreement vs knowledge-base size (random subsets of the 200 seeds; chance = 0.352)
   25 seed rows | k=1: 0.485 | k=5: 0.446  (spread over 30 subsets: k=5 0.405..0.487)
   50 seed rows | k=1: 0.502 | k=5: 0.478  (spread over 30 subsets: k=5 0.433..0.516)
  100 seed rows | k=1: 0.508 | k=5: 0.490  (spread over 30 subsets: k=5 0.465..0.502)
  200 seed rows | k=1: 0.504 | k=5: 0.497


In [ ]:
names = ["neutral", "positive", "negative"]
for c in range(3):
    m = Q_y == c
    print(f"{names[c]:9s} queries: top-5 neighbours with the same label {np.mean(S_y[NB_bench[m, :5]] == c):.3f} (chance {np.mean(S_y == c):.3f})")

neutral   queries: top-5 neighbours with the same label 0.236 (chance 0.220)
positive  queries: top-5 neighbours with the same label 0.571 (chance 0.390)
negative  queries: top-5 neighbours with the same label 0.575 (chance 0.390)


In [ ]:
import os, json
import numpy as np, pandas as pd, torch
from scipy.optimize import minimize
from scipy.special import log_softmax
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

N = np.load(f"{CKPT_DIR}/step4_neighbours.npz")
NB_seed = N["NB_seed"]
S_text, S_y = seed_train["sentence"].tolist(), seed_train["sentiment"].values
llm_tokenizer.padding_side = "left"

def shots_for(nb_row, k):
    """k retrieved examples, least similar first, so the most similar one sits right above the question."""
    return [(S_text[j], int(S_y[j])) for j in nb_row[:k][::-1]]

@torch.no_grad()
def logprobs_per_row(sentences, examples_per_row, tag, batch_size=8, chunk=256):
    """Log-scores (n,3), each sentence with its OWN examples. Every finished chunk is saved to Drive."""
    parts = []
    for j in range(0, len(sentences), chunk):
        path = f"{CKPT_DIR}/step4_{tag}_{j // chunk:03d}.npy"
        if os.path.exists(path):
            parts.append(np.load(path)); continue
        texts = [make_chat_text(s, ex) for s, ex in zip(sentences[j:j + chunk], examples_per_row[j:j + chunk])]
        order = np.argsort([len(t) for t in texts])                 # similar lengths together -> less padding
        out = np.zeros((len(texts), 3), dtype=np.float32)
        for i in range(0, len(texts), batch_size):
            idx = order[i:i + batch_size]
            enc = llm_tokenizer([texts[t] for t in idx], return_tensors="pt", padding=True).to(llm_model.device)
            logits = llm_model(**enc).logits[:, -1, :]
            out[idx] = torch.log_softmax(logits[:, LABEL_TOKEN_IDS].float(), dim=-1).cpu().numpy()
        np.save(path, out); parts.append(out)
        print(f"  {tag}: {min(j + chunk, len(sentences))}/{len(sentences)}")
    return np.concatenate(parts)

def shift(LP, b): return log_softmax(LP + np.array([b[0], 0.0, b[1]]), axis=1)
def fit(LP, y):
    ce = lambda b: -shift(LP, b)[np.arange(len(y)), y].mean()
    return minimize(ce, x0=np.zeros(2), method="Nelder-Mead").x
def cv_cal_acc(LP, y):                                              # honest accuracy of "calibrated" inside the seed set
    hit = np.zeros(len(y), bool)
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=17).split(LP, y):
        hit[te] = shift(LP[te], fit(LP[tr], y[tr])).argmax(1) == y[te]
    return hit.mean()

KS = (4, 8, 12)
res = {}
for k in KS:
    ex = [shots_for(NB_seed[i], k) for i in range(len(S_text))]     # row i never retrieves itself (leave-one-out)
    res[k] = logprobs_per_row(S_text, ex, f"ret_k{k}_seed")
np.savez(f"{CKPT_DIR}/step4_seed_retrieved.npz", **{f"k{k}": v for k, v in res.items()})

zs = np.load(f"{CKPT_DIR}/step2_logprobs.npz", allow_pickle=True)["LP_seed"]
print(f"\nreference, zero-shot on the same 200 seed rows: raw accuracy {np.mean(zs.argmax(1) == S_y):.3f}\n")
accs = {}
for k in KS:
    p = res[k].argmax(1); accs[k] = np.mean(p == S_y)
    print(f"retrieved k={k:2d} | raw accuracy {accs[k]:.3f} | macro-F1 {f1_score(S_y, p, average='macro'):.3f} | "
          f"says-neutral {np.mean(p == 0):.0%} (gold {np.mean(S_y == 0):.0%}) | calibrated accuracy, 5-fold CV {cv_cal_acc(res[k], S_y):.3f}")

best = max(accs.values())
chosen = min(k for k in KS if accs[k] >= best - 0.02)               # rule fixed in advance: smallest k within 2 points of the best
print(f"\nrule: best raw accuracy {best:.3f}; smallest k within 2 points of it -> k = {chosen}")
json.dump({"chosen_k": int(chosen)}, open(f"{CKPT_DIR}/step4_chosen_k.json", "w"))

  ret_k4_seed: 200/200
  ret_k8_seed: 200/200
  ret_k12_seed: 200/200

reference, zero-shot on the same 200 seed rows: raw accuracy 0.650

retrieved k= 4 | raw accuracy 0.730 | macro-F1 0.715 | says-neutral 30% (gold 22%) | calibrated accuracy, 5-fold CV 0.715
retrieved k= 8 | raw accuracy 0.730 | macro-F1 0.717 | says-neutral 30% (gold 22%) | calibrated accuracy, 5-fold CV 0.710
retrieved k=12 | raw accuracy 0.755 | macro-F1 0.739 | says-neutral 26% (gold 22%) | calibrated accuracy, 5-fold CV 0.755

rule: best raw accuracy 0.755; smallest k within 2 points of it -> k = 12


In [ ]:
import os, json
import numpy as np, pandas as pd, torch

N = np.load(f"{CKPT_DIR}/step4_neighbours.npz")
NB_bench, RND_bench, RND_seed = N["NB_bench"], N["RND_bench"], N["RND_seed"]
bench = pd.read_csv(f"{CKPT_DIR}/annotator_benchmark.csv")
Q_text, Q_y = bench["sentence"].tolist(), bench["gold"].values
S_text, S_y = seed_train["sentence"].tolist(), seed_train["sentiment"].values
K = json.load(open(f"{CKPT_DIR}/step4_chosen_k.json"))["chosen_k"]
print("chosen k:", K)
llm_tokenizer.padding_side = "left"

def shots_for(nb_row, k):
    """k examples from the seed rows; for the retrieved version the most similar one comes last (right above the question)."""
    return [(S_text[j], int(S_y[j])) for j in nb_row[:k][::-1]]

@torch.no_grad()
def logprobs_per_row(sentences, examples_per_row, tag, batch_size=8, chunk=256):
    parts = []
    for j in range(0, len(sentences), chunk):
        path = f"{CKPT_DIR}/step4_{tag}_{j // chunk:03d}.npy"
        if os.path.exists(path):
            parts.append(np.load(path)); continue
        texts = [make_chat_text(s, ex) for s, ex in zip(sentences[j:j + chunk], examples_per_row[j:j + chunk])]
        order = np.argsort([len(t) for t in texts])
        out = np.zeros((len(texts), 3), dtype=np.float32)
        for i in range(0, len(texts), batch_size):
            idx = order[i:i + batch_size]
            enc = llm_tokenizer([texts[t] for t in idx], return_tensors="pt", padding=True).to(llm_model.device)
            logits = llm_model(**enc).logits[:, -1, :]
            out[idx] = torch.log_softmax(logits[:, LABEL_TOKEN_IDS].float(), dim=-1).cpu().numpy()
        np.save(path, out); parts.append(out)
        print(f"  {tag}: {min(j + chunk, len(sentences))}/{len(sentences)}")
    return np.concatenate(parts)

LP_ret_s = np.load(f"{CKPT_DIR}/step4_seed_retrieved.npz")[f"k{K}"]            # already scored in Part 2 (leave-one-out)
LP_ret_b = logprobs_per_row(Q_text, [shots_for(NB_bench[i], K) for i in range(len(Q_text))], f"ret_k{K}_bench")
LP_rnd_b = logprobs_per_row(Q_text, [shots_for(RND_bench[i], K) for i in range(len(Q_text))], f"rnd_k{K}_bench")
LP_rnd_s = logprobs_per_row(S_text, [shots_for(RND_seed[i], K) for i in range(len(S_text))], f"rnd_k{K}_seed")
np.savez(f"{CKPT_DIR}/step4_final_k{K}.npz", LP_ret_s=LP_ret_s, LP_ret_b=LP_ret_b, LP_rnd_s=LP_rnd_s, LP_rnd_b=LP_rnd_b)
print("saved.")
for nm, LP in [("retrieved", LP_ret_b), ("random examples", LP_rnd_b)]:
    print(f"raw accuracy on benchmark, {nm}: {np.mean(LP.argmax(1) == Q_y):.3f}")
print(f"raw accuracy on seed rows, random examples: {np.mean(LP_rnd_s.argmax(1) == S_y):.3f}  (retrieved: {np.mean(LP_ret_s.argmax(1) == S_y):.3f})")

chosen k: 12
  rnd_k12_bench: 1536/1707
  rnd_k12_bench: 1707/1707
  rnd_k12_seed: 200/200
saved.
raw accuracy on benchmark, retrieved: 0.664
raw accuracy on benchmark, random examples: 0.637
raw accuracy on seed rows, random examples: 0.720  (retrieved: 0.755)


In [ ]:
import json
import numpy as np, pandas as pd
from scipy.optimize import minimize
from scipy.special import log_softmax
from scipy.stats import binomtest
from sklearn.metrics import f1_score, recall_score

K = json.load(open(f"{CKPT_DIR}/step4_chosen_k.json"))["chosen_k"]
z = np.load(f"{CKPT_DIR}/step2_logprobs.npz", allow_pickle=True)
r = np.load(f"{CKPT_DIR}/step4_final_k{K}.npz")
ys_all, yb, src = z["y_seed"].astype(int), z["y_bench"].astype(int), z["source_bench"].astype(str)

def shift(LP, b): return log_softmax(LP + np.array([b[0], 0.0, b[1]]), axis=1)
def fit(LP, y):
    ce = lambda b: -shift(LP, b)[np.arange(len(y)), y].mean()
    return minimize(ce, x0=np.zeros(2), method="Nelder-Mead").x

preds = {}                                                           # name -> list of prediction vectors (3 for fixed few-shot, else 1)
b = fit(z["LP_seed"], ys_all)
preds["zero-shot"] = [z["LP_bench"].argmax(1)]
preds["zero-shot + cal"] = [shift(z["LP_bench"], b).argmax(1)]
fs_raw, fs_cal = [], []
for dr in (17, 18, 19):                                              # fixed few-shot: each draw calibrated on its own held-out seed rows
    f = np.load(f"{CKPT_DIR}/step3_fs_k4_s{dr}.npz", allow_pickle=True); keep = f["keep_seed"]
    fs_raw.append(f["LP_bench"].argmax(1)); fs_cal.append(shift(f["LP_bench"], fit(f["LP_seed"][keep], ys_all[keep])).argmax(1))
preds["fixed few-shot (3 draws)"], preds["fixed few-shot + cal (3 draws)"] = fs_raw, fs_cal
preds[f"retrieved k={K}"] = [r["LP_ret_b"].argmax(1)]
preds[f"retrieved k={K} + cal"] = [shift(r["LP_ret_b"], fit(r["LP_ret_s"], ys_all)).argmax(1)]
preds[f"random examples k={K}"] = [r["LP_rnd_b"].argmax(1)]
preds[f"random examples k={K} + cal"] = [shift(r["LP_rnd_b"], fit(r["LP_rnd_s"], ys_all)).argmax(1)]
P = lambda name: preds[name][0]

def metrics(plist, m):
    rows = [[np.mean(p[m] == yb[m]), f1_score(yb[m], p[m], average="macro"), np.mean(p[m] == 0),
             *recall_score(yb[m], p[m], average=None, labels=[0, 1, 2])] for p in plist]
    return np.mean(rows, axis=0)

rng = np.random.default_rng(17)
def compare(a, b_, m):
    oa, ob = P(a)[m] == yb[m], P(b_)[m] == yb[m]
    d = ob.astype(int) - oa.astype(int); n = len(d)
    lo, hi = np.percentile([d[rng.integers(0, n, n)].mean() for _ in range(2000)], [2.5, 97.5])
    w, l = int((d == 1).sum()), int((d == -1).sum())
    p = binomtest(w, w + l, 0.5).pvalue if w + l else 1.0
    return f"{d.mean():+.3f} (CI {lo:+.3f}..{hi:+.3f}) fixed {w} broke {l} p={p:.3g}"
def vs_fixed(name, m):                                               # change against each of the 3 fixed few-shot draws
    base = fs_raw if "cal" not in name else fs_cal
    return " ".join(f"{np.mean(P(name)[m] == yb[m]) - np.mean(p[m] == yb[m]):+.3f}" for p in base)

def report(m, title):
    print(f"\n{title}  n={m.sum()}  (gold neutral {np.mean(yb[m] == 0):.0%})")
    print(f"   {'variant':34s} acc    macroF1 says-neu | recall: neutral positive negative")
    for k, pl in preds.items():
        a = metrics(pl, m)
        print(f"   {k:34s} {a[0]:.3f}  {a[1]:.3f}   {a[2]:.0%}     |         {a[3]:.2f}    {a[4]:.2f}     {a[5]:.2f}")
    R, C = f"retrieved k={K}", f"random examples k={K}"
    print(f"   retrieved vs random examples        : {compare(C, R, m)}")
    print(f"   retrieved+cal vs random+cal         : {compare(C + ' + cal', R + ' + cal', m)}")
    print(f"   retrieved vs zero-shot+cal          : {compare('zero-shot + cal', R, m)}")
    print(f"   retrieved, change vs each fixed few-shot draw (17,18,19)    : {vs_fixed(R, m)}")
    print(f"   retrieved+cal, change vs each fixed few-shot+cal draw       : {vs_fixed(R + ' + cal', m)}")

report(np.ones(len(yb), bool), "ALL")
for s in ["random", "margin", "entropy"]:
    report(src == s, s)
out = pd.DataFrame({"source": src, "gold": yb, **{k.replace(" ", "_"): v[0] for k, v in preds.items() if len(v) == 1}})
out.to_csv(f"{CKPT_DIR}/step4_k{K}_predictions.csv", index=False)


ALL  n=1707  (gold neutral 23%)
   variant                            acc    macroF1 says-neu | recall: neutral positive negative
   zero-shot                          0.619  0.617   42%     |         0.77    0.56     0.59
   zero-shot + cal                    0.670  0.638   17%     |         0.42    0.74     0.74
   fixed few-shot (3 draws)           0.677  0.656   24%     |         0.56    0.61     0.80
   fixed few-shot + cal (3 draws)     0.686  0.659   19%     |         0.47    0.73     0.77
   retrieved k=12                     0.664  0.645   25%     |         0.55    0.62     0.77
   retrieved k=12 + cal               0.665  0.634   19%     |         0.43    0.73     0.74
   random examples k=12               0.637  0.623   32%     |         0.63    0.53     0.75
   random examples k=12 + cal         0.665  0.639   20%     |         0.47    0.69     0.75
   retrieved vs random examples        : +0.027 (CI +0.008..+0.046) fixed 160 broke 114 p=0.00645
   retrieved+cal vs random+